In [ ]:
import os
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import xml.etree.ElementTree as ET

class RDD2022Dataset(Dataset):
    """
    Dataset for RDD2022 that reads XML annotations and corresponding images.
    
    This version preserves the aspect ratio by letterboxing the image to 300x300,
    and adjusts the bounding boxes accordingly.
    """
    def __init__(self, root_dir, transform=None, target_size=(300, 300)):
        """
        Args:
            root_dir (str): Root directory of the dataset.
            transform (callable, optional): Additional transformations to apply after resizing.
            target_size (tuple): Desired output image size (width, height).
        """
        self.root_dir = root_dir
        self.target_size = target_size
        self.target_width, self.target_height = target_size

        # Default transform: convert to tensor and normalize.
        self.transform = transform or transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

        # Pre-define label map to avoid recreating it for every sample.
        self.label_map = {
            'D00': 1,  # Linear crack, longitudinal, wheel mark part
            'D10': 2,  # Linear crack, lateral, equal interval
            'D20': 3,  # Alligator crack
            'D40': 4,  # Rutting, pothole, separation
        }

        # Collect all XML files with annotations from each country's folder.
        self.all_xml_files = []
        countries = [d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))]
        for country in countries:
            annot_dir = os.path.normpath(os.path.join(root_dir, country, country, 'train', 'annotations/xmls'))
            if os.path.exists(annot_dir):
                xml_files = [f for f in os.listdir(annot_dir) if f.endswith('.xml')]
                print(f'Found {len(xml_files)} annotations for {country}')
                self.all_xml_files.extend([(country, f) for f in xml_files])

    def __len__(self):
        return len(self.all_xml_files)

    def __getitem__(self, idx):
        country, xml_file = self.all_xml_files[idx]
        xml_path = os.path.join(self.root_dir, country, country, 'train', 'annotations/xmls', xml_file)

        # Parse XML annotation.
        tree = ET.parse(xml_path)
        root_xml = tree.getroot()

        # Get image filename and path.
        img_name = root_xml.find('filename').text
        img_path = os.path.join(self.root_dir, country, country, 'train', 'images', img_name)

        # Load image.
        image = Image.open(img_path).convert('RGB')
        orig_width, orig_height = image.size

        # Parse object annotations.
        boxes = []
        labels = []
        for obj in root_xml.findall('object'):
            label = obj.find('name').text
            # Map alternative labels to the defined ones.
            if label not in self.label_map:
                if label == "D01":
                    label = "D00"
                elif label == "D11":
                    label = "D10"
                else:
                    label = "D40"            

            bbox = obj.find('bndbox')
            xmin = float(bbox.find('xmin').text)
            ymin = float(bbox.find('ymin').text)
            xmax = float(bbox.find('xmax').text)
            ymax = float(bbox.find('ymax').text)
            
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(self.label_map[label])
            else:
                print(f"Skipping degenerate box in {xml_path}")
                

        boxes = torch.FloatTensor(boxes)
        labels = torch.LongTensor(labels)
        
        # Ensure that even images with no objects have a valid target.
        if boxes.numel() == 0:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)

        # -------------------------
        # Preserve Aspect Ratio via Letterboxing:
        # -------------------------
        # Compute scaling factor.
        scale = min(self.target_width / orig_width, self.target_height / orig_height)
        new_width = int(orig_width * scale)
        new_height = int(orig_height * scale)

        # Resize image with preserved aspect ratio.
        resized_image = image.resize((new_width, new_height), resample=Image.BILINEAR)

        # Create a new image of target size and paste the resized image at center.
        new_image = Image.new('RGB', (self.target_width, self.target_height), (0, 0, 0))
        pad_x = (self.target_width - new_width) // 2
        pad_y = (self.target_height - new_height) // 2
        new_image.paste(resized_image, (pad_x, pad_y))

        # Adjust bounding boxes: scale then add the padding offset.
        if boxes.numel() > 0:
            boxes[:, [0, 2]] = boxes[:, [0, 2]] * scale + pad_x
            boxes[:, [1, 3]] = boxes[:, [1, 3]] * scale + pad_y

        # Apply remaining transforms (convert to tensor and normalize).
        image_tensor = self.transform(new_image)

        # Create a target dictionary.
        target = {
            "boxes": boxes,
            "labels": labels
        }
        
        return image_tensor, target

# Define a custom collate function to handle batches with variable-size annotations.
def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

# Create dataset instance.
dataset = RDD2022Dataset(root_dir='../data/RDD2022/RDD2022_all_countries')
# Split dataset into training and validation sets.
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

# #Subsample for faster training
# train_dataset, _ = random_split(train_dataset, [400, len(train_dataset) - 400])
# val_dataset, _ = random_split(val_dataset, [100, len(val_dataset) - 100])

# Create dataloaders.
train_dataloader = DataLoader(train_dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)
val_dataloader = DataLoader(val_dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)


Found 2401 annotations for China_Drone
Found 1977 annotations for China_MotorBike
Found 2829 annotations for Czech
Found 7706 annotations for India
Found 10506 annotations for Japan
Found 8161 annotations for Norway
Found 4805 annotations for United_States


In [2]:
import os
import time
import torch
import numpy as np
from torchvision.ops import nms
from sklearn.metrics import average_precision_score

def compute_iou_np(box, boxes):
    """Compute IoU between one box and an array of boxes."""
    xA = np.maximum(box[0], boxes[:, 0])
    yA = np.maximum(box[1], boxes[:, 1])
    xB = np.minimum(box[2], boxes[:, 2])
    yB = np.minimum(box[3], boxes[:, 3])
    interW = np.maximum(0, xB - xA)
    interH = np.maximum(0, yB - yA)
    interArea = interW * interH
    boxArea = (box[2] - box[0]) * (box[3] - box[1])
    boxesArea = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    unionArea = boxArea + boxesArea - interArea + 1e-6
    return interArea / unionArea

def evaluate_model(model, dataloader, num_classes, iou_threshold=0.5, score_threshold=0.5, epoch=None, log_dir=None):
    """
    Evaluates the SSD model on the given dataloader.
    
    Computes:
      - mAP (mean Average Precision) via sklearn's average_precision_score,
      - per-class precision, recall, and F1 score (computed using TP/FP/FN),
      - average inference time per image.
      
    Assumes background is labeled as 0 and object classes are 1...num_classes-1.
    
    :param model: Trained PyTorch model.
    :param dataloader: Dataloader for validation/test set.
    :param num_classes: Total number of classes (including background).
    :param iou_threshold: IoU threshold for a true positive.
    :param score_threshold: Minimum score to keep a detection.
    :param epoch: (Optional) Epoch number for logging.
    :param log_dir: (Optional) Directory to log metrics.
    :return: mAP, average precision, recall, F1 score, and inference time per image.
    """
    model.eval()
    device = next(model.parameters()).device

    # Dictionaries to hold predictions and ground truths per class.
    detections = {c: [] for c in range(1, num_classes)}
    ground_truths = {c: {} for c in range(1, num_classes)}

    total_inference_time = 0.0
    image_count = 0

    with torch.no_grad():
        for images, targets in dataloader:
            images_tensor = torch.stack(images).to(device)
            start_time = time.time()
            outputs = model(images_tensor)
            total_inference_time += time.time() - start_time
            batch_size = len(outputs)
            
            for i in range(batch_size):
                curr_img_id = image_count + i
                target = targets[i]
                # Process ground truth boxes.
                gt_boxes = target["boxes"].cpu().numpy()
                gt_labels = target["labels"].cpu().numpy()
                for cls in np.unique(gt_labels):
                    cls = int(cls)
                    if cls == 0:
                        continue
                    inds = np.where(gt_labels == cls)[0]
                    if curr_img_id not in ground_truths[cls]:
                        ground_truths[cls][curr_img_id] = []
                    for box in gt_boxes[inds]:
                        ground_truths[cls][curr_img_id].append(box)
                
                # Process model predictions.
                output = outputs[i]
                pred_boxes = output["boxes"].cpu()
                pred_labels = output["labels"].cpu()
                pred_scores = output["scores"].cpu()
                
                # Only keep predictions above the score threshold.
                keep = pred_scores > score_threshold
                pred_boxes = pred_boxes[keep].numpy()
                pred_labels = pred_labels[keep].numpy()
                pred_scores = pred_scores[keep].numpy()
                
                for j in range(len(pred_boxes)):
                    cls = int(pred_labels[j])
                    if cls == 0:
                        continue
                    detections[cls].append((curr_img_id, pred_scores[j], pred_boxes[j]))
            image_count += batch_size

    # Compute per-class metrics.
    ap_per_class = {}
    precision_per_class = {}
    recall_per_class = {}
    f1_per_class = {}

    for cls in range(1, num_classes):
        cls_dets = detections.get(cls, [])
        # Total number of ground truth boxes for this class.
        num_gt = sum(len(boxes) for boxes in ground_truths.get(cls, {}).values())
        # Track which ground truth boxes have been detected per image.
        gt_detected = {img_id: np.zeros(len(boxes), dtype=bool)
                       for img_id, boxes in ground_truths.get(cls, {}).items()}
        y_scores = []
        y_true = []
        TP = 0
        FP = 0

        # Sort detections by descending score.
        for det in sorted(cls_dets, key=lambda x: x[1], reverse=True):
            img_id, score, box = det
            y_scores.append(score)
            if img_id in ground_truths.get(cls, {}):
                gt_boxes = np.array(ground_truths[cls][img_id])
                ious = compute_iou_np(box, gt_boxes)
                max_iou = 0
                max_idx = -1
                if ious.size > 0:
                    max_iou = np.max(ious)
                    max_idx = np.argmax(ious)
                if max_iou >= iou_threshold and not gt_detected[img_id][max_idx]:
                    y_true.append(1)
                    TP += 1
                    gt_detected[img_id][max_idx] = True
                else:
                    y_true.append(0)
                    FP += 1
            else:
                y_true.append(0)
                FP += 1

        # Compute precision, recall, and F1.
        if len(y_true) == 0:
            precision_cls = 0.0
            recall_cls = 0.0
            f1_cls = 0.0
            ap_cls = 0.0
        else:
            precision_cls = TP / (TP + FP) if (TP + FP) > 0 else 0.0
            recall_cls = TP / num_gt if num_gt > 0 else 0.0
            f1_cls = 2 * precision_cls * recall_cls / (precision_cls + recall_cls) if (precision_cls + recall_cls) > 0 else 0.0
            try:
                ap_cls = average_precision_score(y_true, y_scores)
            except Exception:
                ap_cls = 0.0

        ap_per_class[cls] = ap_cls
        precision_per_class[cls] = precision_cls
        recall_per_class[cls] = recall_cls
        f1_per_class[cls] = f1_cls

    # Aggregate metrics across classes.
    mAP = np.mean(list(ap_per_class.values())) if ap_per_class else 0.0
    avg_precision = np.mean(list(precision_per_class.values())) if precision_per_class else 0.0
    avg_recall = np.mean(list(recall_per_class.values())) if recall_per_class else 0.0
    avg_f1 = np.mean(list(f1_per_class.values())) if f1_per_class else 0.0
    avg_inference_time = total_inference_time / (image_count + 1e-6)

    if epoch is not None:
        print(f"Epoch {epoch}: mAP: {mAP:.4f}, Precision: {avg_precision:.4f}, "
              f"Recall: {avg_recall:.4f}, F1: {avg_f1:.4f}, Inference Time: {avg_inference_time:.4f} sec")
    if log_dir is not None and epoch is not None:
        os.makedirs(log_dir, exist_ok=True)
        log_file = os.path.join(log_dir, "MobileNetV3_small_metrics.csv")
        write_header = not os.path.exists(log_file)
        with open(log_file, "a") as f:
            if write_header:
                f.write("epoch,mAP,precision,recall,f1,inference_time\n")
            f.write(f"{epoch},{mAP},{avg_precision},{avg_recall},{avg_f1},{avg_inference_time}\n")

    return mAP, avg_precision, avg_recall, avg_f1, avg_inference_time


In [3]:
import torch
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.resnet import resnet50
from torchvision.models.mobilenet import mobilenet_v3_small
from torchvision.models.detection.ssd import SSD

# for learning_rate in np.arange(0.0008, 0.01, 0.001):
#     for momentum in np.arange(0.7, 1.0, 0.1):
#         for conf_threshold in np.arange(0.3, 0.7, 0.1):
#             for iou_threshold in np.arange(0.3, 0.7, 0.1):            
#                 best_setup = {'learning_rate': learning_rate, 'momentum': momentum, 'conf_threshold': conf_threshold, 'iou_threshold': iou_threshold}
#                 print(f"Training with learning_rate: {learning_rate}, momentum: {momentum},  conf_threshold: {conf_threshold}, iou_threshold: {iou_threshold}")
#                 best_f1_score = 0      
                
best_hyperparameters = None
with open("../SSD_runs/best_hyperparameters.txt", "r") as f:
    best_hyperparameters = dict(eval(f.read()))

learning_rate = best_hyperparameters['learning_rate']
momentum = best_hyperparameters['momentum']
conf_threshold = best_hyperparameters['conf_threshold']
iou_threshold = best_hyperparameters['iou_threshold']          
# -------------------------------
# Training Loop
# -------------------------------                
num_classes = 5  # Example: 4 defect classes + 1 background (adjust as needed)
# Initialize SSD model (assuming the SSD class supports a custom backbone).

# Create an anchor generator. The sizes and aspect ratios here are examples based on SSD300.
anchor_generator = AnchorGenerator(
    sizes=((21, 45, 99, 153, 207, 261, 315),),  # A tuple for each feature map level.
    aspect_ratios=((0.5, 1.0, 2.0),)
)

# Define the input image size expected by SSD (e.g., 300 for SSD300)
input_size = (300, 300)

resnet50_backbone = resnet50(pretrained=True)
# Remove the fully connected layers to get feature maps.
resnet50_backbone = nn.Sequential(*list(resnet50_backbone.children())[:-2])
resnet50_backbone.out_channels = [2048] # ResNet50's last conv layer outputs 2048 channels.

mobilenet_backbone = mobilenet_v3_small(pretrained=True)
mobilenet_backbone = nn.Sequential(*list(mobilenet_backbone.children())[:-1])   # Remove the classifier layer.
mobilenet_backbone.out_channels = [576]  # MobileNetV3's last conv layer outputs 576 channels.

model = SSD(num_classes=num_classes, backbone=mobilenet_backbone, anchor_generator=anchor_generator, size=input_size)
model = model.to('cuda' if torch.cuda.is_available() else 'cpu')

# Define device and move the model to it.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Initialize optimizer.
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum, weight_decay=0.001)

# Loss functions.
loc_criterion = nn.SmoothL1Loss()         # Localization loss.
conf_criterion = nn.CrossEntropyLoss()  # Classification loss.

num_epochs = 50  # Set desired number of epochs.
log_dir = "../SSD_runs/logs"  # Directory to log metrics.

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for images, targets in tqdm(train_dataloader, desc=f"Epoch {epoch+1} Training"):
        # Stack images and move them to the device.
        images = torch.stack(images).to(device)
        # Move targets to the device.
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        optimizer.zero_grad()
        
        # Forward pass: pass both images and targets.
        loss_dict = model(images, targets)
        if not isinstance(loss_dict, dict):         
            print(loss_dict)
            raise ValueError("model(images, targets) should return a dict of losses")
        loss = sum(loss for loss in loss_dict.values())
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()


    avg_epoch_loss = epoch_loss / len(train_dataloader)
    print(f"Epoch {epoch+1} Training Loss: {avg_epoch_loss:.4f}")

    # Evaluate and log metrics after the epoch.
    # Ensure evaluate_model is defined elsewhere.
    mAP, avg_precision, avg_recall, avg_f1, avg_inference_time = evaluate_model(
        model, 
        val_dataloader, 
        num_classes=num_classes, 
        epoch=epoch+1, 
        log_dir=log_dir,
        iou_threshold=iou_threshold,
        score_threshold=conf_threshold
    )                        
    print(f"[Epoch {epoch+1}] mAP: {mAP:.4f}, F1_Score: {avg_f1:.4f}, Precision: {avg_precision:.4f}, Recall: {avg_recall:.4f}, Inference: {avg_inference_time:.4f}\n")

# if avg_f1 > best_f1_score:
#     best_f1_score = avg_f1
#     best_setup = {'learning_rate': learning_rate, 'momentum': momentum, 'conf_threshold': conf_threshold, 'iou_threshold': iou_threshold}
#     with open("../SSD_runs/best_hyperparameters.txt", "w") as f:
#         f.write(str(best_setup))
        
# Save the final model.
torch.save(model.state_dict(), '../SSD_runs/SSD_MobileNetV3_small_model.pth')
print("Training completed.")


c:\Users\raivo\Desktop\final_paper\kood\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\raivo\Desktop\final_paper\kood\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
c:\Users\raivo\Desktop\final_paper\kood\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. Y

KeyboardInterrupt: 